# Della GPU Power

## Setup

The pandas Python package is required to use this notebook. The next line downloads a necessary Python module:

#!wget https://raw.githubusercontent.com/PrincetonUniversity/job_defense_shield/refs/heads/main/src/job_defense_shield/efficiency.py

In [1]:
import os
import re
import requests
import subprocess
from datetime import datetime
import pandas as pd
from efficiency import get_stats_dict
from efficiency import get_detailed_metric
from efficiency import cpu_memory_usage
from efficiency import gpu_memory_usage_eff_tuples

In [2]:
# convert slurm timestamps to seconds
os.environ["SLURM_TIME_FORMAT"] = "%s"

In [3]:
def get_data_from_sacct(clusters: str,
                        partition: str,
                        start_date: str,
                        end_date: str,
                        fields: str) -> pd.DataFrame:
    """Return a dataframe of the sacct output."""
    partition = f"-r {partition}" if partition else partition
    cmd = f"sacct -M {clusters} {partition} -a -X -P -n -S {start_date} -E {end_date} -o {fields}"
    output = subprocess.run(cmd,
                            stdout=subprocess.PIPE,
                            shell=True,
                            timeout=100,
                            text=True,
                            check=True)
    rows = output.stdout.splitlines()
    cols = fields.split(",")
    df = pd.DataFrame([row.split("|")[:len(cols)]
                       for row in rows if row.count("|") > len(cols) - 2])
    df.columns = cols
    return df

## The job data is obtained and cleaned

Enter the cluster name, a comma-separated list of the partitions of interest, and the time window:

In [4]:
clusters = "della"
partition = "ailab,ailab-p,cryoem,gpu,gputest,gpu-ee,gpu-wentzlaff,grace,mig,pli-c,pli,pli-lc,pli-p,rtx6000"  # use an empty string for all partitions
start_date = "2026-10-21"
end_date = "2026-10-27"  # do not use "now"

You can skip to the last section of the notebook.

In [5]:
fields = "jobid,cluster,user,partition,state,account,ncpus,elapsedraw,alloctres,admincomment"
df = get_data_from_sacct(clusters, partition, start_date, end_date, fields)

In [6]:
df = df[pd.notna(df.elapsedraw)]
df = df[df.elapsedraw.str.isnumeric()]
df.elapsedraw = df.elapsedraw.astype("int64")
df = df[df.elapsedraw > 0]

In [7]:
df = df[pd.notna(df.ncpus)]
df = df[df.ncpus.str.isnumeric()]
df.ncpus = df.ncpus.astype("int64")
df = df[df.ncpus > 0]

## New fields are added

In [8]:
def gpus_per_job(tres: str) -> int:
    """Return the number of allocated GPUs."""
    gpus = re.findall(r"gres/gpu=\d+", tres)
    return int(gpus[0].replace("gres/gpu=", "")) if gpus else 0

In [9]:
df["gpus"] = df.alloctres.apply(gpus_per_job)
df["gpu-seconds"] = df.apply(lambda row: row["elapsedraw"] * row["gpus"], axis='columns')
df["gpu-hours"] = df["gpu-seconds"] / 3600
df["admincomment"] = df["admincomment"].apply(get_stats_dict)

In [10]:
# could reduce mig by 1/7th
by_partition = df.groupby("partition").agg({"gpu-hours": "sum", "user": "nunique"})
by_partition.sort_values(by="gpu-hours", ascending=False).round()

,gpu-hours,user
partition,,
gpu,59367.0,137
pli-c,55073.0,31
ailab,23216.0,63
cryoem,12759.0,22
gputest,7805.0,192
mig,7615.0,49
pli,1514.0,31
rtx6000,1042.0,10
pli-lc,924.0,3


Sanity check: there are 336 GPUs for PLI

In [11]:
# days = end_date - start_date = 7
x = by_partition.reset_index()
x[x.partition.str.contains("pli")]["gpu-hours"].sum() / (336 * 24 * 7)

np.float64(1.0188282824703954)

In [12]:
df["power-raw"] = df.apply(lambda row: get_detailed_metric(row["admincomment"],
                                                           "power_usage_milliwatts_avg",
                                                           row["jobid"],
                                                           row["cluster"],
                                                           verbose=False),
                                                           axis="columns")
cols = ["power-tuple", "error_code"]
df[cols] = pd.DataFrame(df["power-raw"].tolist(), index=df.index)
df = df[df["error_code"] == 0]

In [13]:
df["power-sum"] = df["power-tuple"].apply(lambda mylist: sum(mylist))

The table below contains simple statistics about your data:

In [14]:
df.describe().round(1)

,ncpus,elapsedraw,gpus,gpu-seconds,gpu-hours,error_code,power-sum
count,52681.0,52681.0,52681.0,52681.0,52681.0,52681.0,52681.0
mean,7.2,7006.2,1.2,11282.5,3.1,0.0,291257.8
std,10.0,20571.0,1.2,54072.2,15.0,0.0,452350.0
min,1.0,57.0,1.0,57.0,0.0,0.0,32586.1
25%,2.0,206.0,1.0,213.0,0.1,0.0,129223.5
50%,4.0,1089.0,1.0,1232.0,0.3,0.0,207126.3
75%,8.0,3507.0,1.0,3905.0,1.1,0.0,288893.3
max,288.0,518719.0,36.0,3458472.0,960.7,0.0,11285741.4


## 1. Total GPU Power

In [15]:
milli_watts_to_watts = 1000
total_power_watts = df["power-sum"].sum().item() / milli_watts_to_watts
total_power_watts

15343751.86

## 2. Power as a Function of Time

See: https://princetonuniversity.github.io/jobstats/setup/summaries/

In [16]:
!sacct -j 16268293 -X -o start,end

              Start                 End 
------------------- ------------------- 
         1793036310          1793069726 


In [17]:
params = {'query': 'nvidia_gpu_power_usage_milliwatts{cluster="della", jobid="16268293"}',
          'start': 1793036310,
          'end': 1793069726,
          'step': "30s"}
response = requests.get('http://vigilant2:8480/api/v1/query_range', params)
print(response.json())

{'status': 'success', 'data': {'resultType': 'matrix', 'result': [{'metric': {'__name__': 'nvidia_gpu_power_usage_milliwatts', 'cluster': 'della', 'instance': 'della-k18g2:9445', 'job': 'Della GPU Nodes', 'jobid': '16268293', 'minor_number': '3', 'name': 'NVIDIA H100 80GB HBM3', 'ordinal': '3', 'service': 'compute', 'userid': '364812', 'uuid': 'GPU-1e80977e-6f69-2bcd-8554-485ff7d5555a'}, 'values': [[1793036340, '72690'], [1793036370, '124206'], [1793036400, '124460'], [1793036430, '123372'], [1793036460, '123010'], [1793036490, '270681'], [1793036520, '270595'], [1793036550, '284842'], [1793036580, '270924'], [1793036610, '267375'], [1793036640, '258008'], [1793036670, '281341'], [1793036700, '283978'], [1793036730, '218567'], [1793036760, '237500'], [1793036790, '268257'], [1793036820, '278029'], [1793036850, '239605'], [1793036880, '286149'], [1793036910, '263044'], [1793036940, '272466'], [1793036970, '268634'], [1793037000, '305273'], [1793037030, '280931'], [1793037060, '275403'],

In [18]:
df.state = df.state.apply(lambda s: "CANCELLED" if "CANCEL" in s else s)
df.state.value_counts()

state
COMPLETED        47625
TIMEOUT           1953
FAILED            1747
CANCELLED         1228
OUT_OF_MEMORY      124
RESIZING             3
REQUEUED             1
Name: count, dtype: int64

NODE_FAIL will appear if it happens.

Downtimes are 6 am to about 2 pm on the second Tuesday of each month. The regularly scheduled downtime for October 2026 took place on October 20th instead of 13th.